# Notebook 14 — Cross-Scale Replication of NB11 on GPT-2-XL
**CS 590NN | Amogh | Apr 25 — does the optimizer-step finding generalize to 1.5B?**

## What we're testing

NB11 found on Qwen3-0.6B that:
- Random-target B edits destroy A as much as real B edits (Wilcoxon p = 0.74, 7/10 vs 8/10 destroyed)
- Dry-run gradients (lr ~ 0) cause zero damage (0/10 destroyed, p = 1.00 vs no edit)
- **The optimizer step is the proximate cause; B's target content is irrelevant.**

This is the strongest finding in the project. But it's only on one model. **Does it generalize to a 2.5x larger model (GPT-2-XL, 1.5B)?**

If yes → upgrades the contribution from "we observed X on small Qwen" to "this is a general property of KL-stabilized editing on transformer LMs."

If no → still informative: the mechanism is scale-dependent, which is itself a notable finding (you already showed in Tables 2-4 that localization × scale interactions matter).

## Setup

Same 4-condition protocol as NB11:
- C0: skip (no edit B at all)
- C1: real (edit B with true target)
- C2: dry_run (gradients with lr ~ 0)
- C3: random_target (edit B against random token)

Same 10 pairs, same edit hyperparameters. Only difference: GPT-2-XL backbone.

## Predictions

| Pattern | Conclusion |
|---|---|
| C2 ≈ C0 AND C3 ≈ C1 | Replicates NB11 — content-independence is general |
| C3 < C1 (real > random) | At larger scale, content matters — finding is small-model-specific |
| C2 > C0 (dry-run causes damage) | At larger scale, even gradients alone perturb — different mechanism |

## Compute

~30-40 min on A100 (GPT-2-XL is 2.5x slower than Qwen). 40 runs total.

### 14.0 Install (run once, restarts)

In [ ]:
import subprocess, sys, os
def pip(args): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + args)
pip(['numpy==1.26.4'])
pip(['transformer-lens', 'transformers', 'datasets', 'accelerate', 'einops', 'jaxtyping'])
pip(['matplotlib', 'scipy'])
print('Restarting...'); os.kill(os.getpid(), 9)

### 14.1 Imports

In [ ]:
import torch, json, requests, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
from scipy import stats
from transformer_lens import HookedTransformer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS_DIR = Path('results_nb14'); RESULTS_DIR.mkdir(exist_ok=True)
FIG_DIR = RESULTS_DIR / 'figures'; FIG_DIR.mkdir(exist_ok=True)
torch.manual_seed(42); random.seed(42); np.random.seed(42)
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    free, total = torch.cuda.mem_get_info()
    print(f'GPU: {torch.cuda.get_device_name(0)} ({free/1e9:.1f}/{total/1e9:.1f} GB free)')

### 14.2 Load GPT-2-XL

In [ ]:
MODEL_NAME = 'gpt2-xl'  # 1.5B params, 48 layers, 25 heads
model = HookedTransformer.from_pretrained(
    MODEL_NAME,
    center_unembed=True, center_writing_weights=False,
    fold_ln=True, refactor_factored_attn_matrices=False, device=DEVICE,
)
model.set_use_attn_result(True)
model.eval()
model.tokenizer.pad_token = model.tokenizer.eos_token
free, _ = torch.cuda.mem_get_info()
print(f'Loaded {MODEL_NAME}: {model.cfg.n_layers}L x {model.cfg.n_heads}H')
print(f'GPU free: {free/1e9:.1f} GB')

### 14.3 Load CounterFact + same 10 pair indices as NB11

In [ ]:
@dataclass
class EditSample:
    idx: int; prompt: str; subject: str
    target_new: str; target_true: str

print('Downloading CounterFact...')
raw = requests.get('https://rome.baulab.info/data/dsets/counterfact.json', timeout=120).json()

def make_sample(idx):
    rr = raw[idx]['requested_rewrite']
    return EditSample(idx=idx, prompt=rr['prompt'].format(rr['subject']),
                      subject=rr['subject'],
                      target_new=' '+rr['target_new']['str'], target_true=' '+rr['target_true']['str'])

# Same pairs as NB11 (auto-fetched there from v3 — we hardcode here for reproducibility)
PAIR_INDICES = [(11, 1221), (231, 2065), (257, 4161), (410, 1895), (431, 1516),
                (8, 547), (584, 737), (750, 753), (781, 799), (861, 877)]
pairs = [{'A': make_sample(a), 'B': make_sample(b)} for a, b in PAIR_INDICES]
print(f'Built {len(pairs)} pairs (same as NB11)')

### 14.4 ROME-trace localization (same as NB11, GPT-2-XL has more layers)

In [ ]:
def rome_trace_top_layers(model, sample, k=5):
    new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0].item()
    true_id = model.to_tokens(sample.target_true, prepend_bos=False)[0,0].item()
    tokens = model.to_tokens(sample.prompt)
    n_tok = tokens.shape[1]
    if n_tok <= 2: return list(range(min(k, model.cfg.n_layers)))
    subj_pos = list(range(1, n_tok-1))
    corrupt = tokens.clone()
    corrupt[0, subj_pos] = torch.randint(1000, model.cfg.d_vocab-1, (len(subj_pos),), device=tokens.device)
    with torch.no_grad():
        cl, cc = model.run_with_cache(tokens)
        kl, _ = model.run_with_cache(corrupt)
    clean_score = (cl[0,-1,true_id] - cl[0,-1,new_id]).item()
    corrupt_score = (kl[0,-1,true_id] - kl[0,-1,new_id]).item()
    eff = max(abs(clean_score - corrupt_score), 0.5)
    effects = []
    for L in range(model.cfg.n_layers):
        hn = f'blocks.{L}.hook_mlp_out'
        if hn not in cc: continue
        ca = cc[hn].clone()
        def hk(v, hook): return ca
        with torch.no_grad():
            patched = model.run_with_hooks(corrupt, fwd_hooks=[(hn, hk)])
        recovery = (patched[0,-1,true_id] - patched[0,-1,new_id]).item() - corrupt_score
        effects.append((L, abs(recovery)/eff))
    effects.sort(key=lambda x: -x[1])
    del cc; torch.cuda.empty_cache()
    return [L for L, _ in effects[:k]]

### 14.5 Edit function (identical to NB11 — only the model changes)

In [ ]:
NEUTRAL = [
    'The sum of two and three is', 'Twelve divided by four equals',
    'The square root of nine is', 'Ten times ten equals',
    'The capital of Japan is', 'The largest ocean on Earth is the',
    'Mount Everest is located in the', 'The Amazon River flows through',
    'Water boils at one hundred degrees', 'The chemical symbol for gold is',
    'Plants produce oxygen through a process called', 'The Earth orbits the',
    'A week contains seven', 'The primary colors are red, blue, and',
    'Humans have two lungs and one', 'A triangle has three',
]

def cache_ref(model, prompts):
    cache = []
    model.eval()
    with torch.no_grad():
        for p in prompts:
            t = model.to_tokens(p)
            ref_lp = torch.log_softmax(model(t)[0,-1,:], dim=-1).detach().clone()
            cache.append((t, ref_lp))
    return cache

def kl_against(model, ref_cache):
    total = 0.0
    for t, ref_lp in ref_cache:
        edit_lp = torch.nn.functional.log_softmax(model(t)[0,-1,:], dim=-1)
        total = total + (ref_lp.exp() * (ref_lp - edit_lp)).sum()
    return total / len(ref_cache)

def get_params(model, mlp_layers):
    params = []
    for L in mlp_layers:
        params += [model.blocks[L].mlp.W_in, model.blocks[L].mlp.W_out]
    return params

def edit(model, sample, mlp_layers, n_steps, lr, beta_kl, ref_cache, mode='real', random_seed=0):
    if mode == 'skip':
        return
    params = get_params(model, mlp_layers)
    if not params:
        params = get_params(model, list(range(model.cfg.n_layers)))
    for p in model.parameters(): p.requires_grad_(False)
    for p in params: p.requires_grad_(True)
    eff_lr = lr if mode != 'dry_run' else 1e-12
    opt = torch.optim.Adam(params, lr=eff_lr)
    new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0]
    true_id = model.to_tokens(sample.target_true, prepend_bos=False)[0,0]
    if mode == 'random_target':
        random.seed(random_seed)
        target_id = torch.tensor(random.randint(1000, model.cfg.d_vocab - 1000), device=DEVICE)
    else:
        target_id = new_id
    tokens = model.to_tokens(sample.prompt)
    for _ in range(n_steps):
        model.train()
        opt.zero_grad(set_to_none=True)
        logits = model(tokens)
        lp = torch.nn.functional.log_softmax(logits[0,-1,:], dim=-1)
        loss = -lp[target_id] + lp[true_id]
        if ref_cache and beta_kl > 0:
            loss = loss + beta_kl * kl_against(model, ref_cache)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
        opt.step()
    for p in model.parameters(): p.requires_grad_(True)
    torch.cuda.empty_cache()

def measure_p_new(model, sample):
    model.eval()
    with torch.no_grad():
        new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0]
        return float(torch.softmax(model(model.to_tokens(sample.prompt))[0,-1,:], dim=-1)[new_id])

def restore(model, state):
    with torch.no_grad():
        for n, p in model.named_parameters(): p.copy_(state[n])
    torch.cuda.empty_cache()

print('Snapshotting weights (this may take a moment for GPT-2-XL)...')
original_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
torch.cuda.empty_cache()
free, _ = torch.cuda.mem_get_info()
print(f'GPU free: {free/1e9:.1f} GB')

### 14.6 The 4-condition loop

In [ ]:
CONDITIONS = ['skip', 'real', 'dry_run', 'random_target']
N_STEPS_A = 5
N_STEPS_B = 3
BETA_KL = 0.1
LR = 5e-3

ROWS_FILE = RESULTS_DIR / f'empty_edit_gpt2xl_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
rows = []
started = datetime.now()
ref_cache = cache_ref(model, NEUTRAL)

all_samples = {s.idx: s for p in pairs for s in (p['A'], p['B'])}
print(f'Pre-computing ROME-trace for {len(all_samples)} unique samples on GPT-2-XL...')
circuits = {idx: rome_trace_top_layers(model, s, k=5) for idx, s in all_samples.items()}
print('Done.')

total_runs = len(pairs) * len(CONDITIONS)
print(f'Total runs: {total_runs}')

run_idx = 0
for p_idx, p in enumerate(pairs):
    A, B = p['A'], p['B']
    for cond in CONDITIONS:
        run_idx += 1
        try:
            restore(model, original_state)
            edit(model, A, circuits[A.idx], n_steps=N_STEPS_A, lr=LR, beta_kl=BETA_KL,
                 ref_cache=ref_cache, mode='real')
            p_A_postA = measure_p_new(model, A)
            edit(model, B, circuits[B.idx], n_steps=N_STEPS_B, lr=LR, beta_kl=BETA_KL,
                 ref_cache=ref_cache, mode=cond, random_seed=p_idx)
            p_A_after = measure_p_new(model, A)
            p_B_after = measure_p_new(model, B)
            A_displaced = max(0, p_A_postA - p_A_after)
            row = {
                'pair_idx': p_idx, 'A_idx': A.idx, 'B_idx': B.idx, 'condition': cond,
                'p_A_postA': p_A_postA, 'p_A_after_B': p_A_after, 'p_B_after_B': p_B_after,
                'A_displaced': A_displaced, 'status': 'ok',
            }
            rows.append(row)
            print(f'[{run_idx:>2}/{total_runs}] pair {p_idx:>2}  cond={cond:>13}  '
                  f'p_A_postA={p_A_postA:.3f}  p_A_after={p_A_after:.3f}  '
                  f'A_displaced={A_displaced:.3f}  p_B_after={p_B_after:.3f}')
            with open(ROWS_FILE, 'w') as f: json.dump({'rows': rows}, f, indent=2)
        except Exception as e:
            print(f'[{run_idx:>2}/{total_runs}] FAILED: {type(e).__name__}: {e}')
            rows.append({'pair_idx': p_idx, 'condition': cond, 'status': 'failed', 'error': str(e)[:200]})
            torch.cuda.empty_cache()

elapsed = (datetime.now() - started).total_seconds() / 60
print(f'Total runtime: {elapsed:.1f} min')
restore(model, original_state)

### 14.7 Aggregate, test, and compare to NB11 baseline

In [ ]:
df = pd.DataFrame([r for r in rows if r.get('status') == 'ok'])
print(f'OK rows: {len(df)}')

agg = df.groupby('condition').agg(
    n=('pair_idx','count'),
    A_displaced_mean=('A_displaced','mean'),
    A_displaced_median=('A_displaced','median'),
    A_displaced_std=('A_displaced','std'),
    p_B_after_mean=('p_B_after_B','mean'),
).round(3)
print('\nPer-condition aggregate (GPT-2-XL):')
print(agg)

# NB11 baseline (Qwen3-0.6B) for comparison
NB11_QWEN = {
    'skip': 0.000, 'real': 0.803, 'dry_run': 0.000, 'random_target': 0.781
}
print('\n=== Comparison to NB11 (Qwen3-0.6B) ===')
print(f'  {"condition":>14}  {"GPT-2-XL":>10}  {"Qwen-0.6B":>10}  diff')
for c in ['skip','real','dry_run','random_target']:
    sub = df[df['condition']==c]
    gpt = sub['A_displaced'].mean()
    qwen = NB11_QWEN[c]
    print(f'  {c:>14}  {gpt:>10.3f}  {qwen:>10.3f}  {gpt-qwen:+.3f}')

print('\n=== Pairwise Wilcoxon (paired) ===')
from itertools import combinations
results = {}
for c1, c2 in combinations(CONDITIONS, 2):
    piv = df.pivot_table(index='pair_idx', columns='condition', values='A_displaced').dropna(subset=[c1, c2])
    if len(piv) >= 4:
        w, p = stats.wilcoxon(piv[c1], piv[c2])
        sig = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns'))
        diff = piv[c1].mean() - piv[c2].mean()
        print(f'  {c1:>13} vs {c2:>13}: p={p:.4f} {sig}  (diff={diff:+.3f})')
        results[(c1,c2)] = {'p': float(p), 'diff': float(diff)}

print('\n=== Verdict ===')
m = {c: df[df['condition']==c]['A_displaced'].mean() for c in CONDITIONS}
for c in CONDITIONS:
    print(f'  {c:>13}: {m[c]:.3f}')

# Specifically check if NB11's headline replicates
real_v_random_p = results.get(('real','random_target'), {}).get('p', None)
dry_v_skip_p = results.get(('skip','dry_run'), {}).get('p', None)
print('\n=== Cross-scale replication check ===')
print(f'NB11 found:  real ≈ random_target (Wilcoxon p = 0.74) AND  dry_run ≈ skip (Wilcoxon p = 1.00)')
if real_v_random_p is not None and dry_v_skip_p is not None:
    rep_content = real_v_random_p > 0.05 and abs(m['real'] - m['random_target']) < 0.15
    rep_dryrun  = dry_v_skip_p > 0.05 and m['dry_run'] < 0.1
    print(f'NB14 found:  real vs random_target p = {real_v_random_p:.4f}, dry vs skip p = {dry_v_skip_p:.4f}')
    print(f'\n  Content-independence (real ≈ random):  {"✓ REPLICATES" if rep_content else "✗ DOES NOT REPLICATE"}')
    print(f'  Dry-run no-damage (dry_run ≈ skip):    {"✓ REPLICATES" if rep_dryrun  else "✗ DOES NOT REPLICATE"}')
    if rep_content and rep_dryrun:
        print('\n  *** NB11 finding GENERALIZES across scale (Qwen 0.6B and GPT-2-XL 1.5B). ***')
    elif rep_content or rep_dryrun:
        print('\n  Partial replication. The mechanism may be scale-dependent in some respects.')
    else:
        print('\n  NO replication. NB11 finding may be specific to small-scale Qwen.')

### 14.8 Cross-scale figure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left — GPT-2-XL only
ax = axes[0]
order = ['skip','dry_run','random_target','real']
labels = ['C0: no edit B', 'C2: gradients (lr~0)', 'C3: random target', 'C1: real edit']
means = [df[df['condition']==c]['A_displaced'].mean() for c in order]
stds  = [df[df['condition']==c]['A_displaced'].std()  for c in order]
colors = ['#888888', '#cc6633', '#3366cc', '#cc3333']
bars = ax.bar(labels, means, yerr=stds, capsize=5, color=colors, edgecolor='black', alpha=0.85)
ax.set_ylabel('A_displaced')
ax.set_title('GPT-2-XL (1.5B)')
ax.grid(alpha=0.3, axis='y'); ax.set_ylim(-0.05, 1.05); ax.tick_params(axis='x', rotation=15)
for bar, mv in zip(bars, means):
    ax.annotate(f'{mv:.2f}', (bar.get_x()+bar.get_width()/2, bar.get_height()+0.02),
                ha='center', fontsize=10, fontweight='bold')

# Right — Cross-scale comparison: GPT-2-XL vs Qwen
ax = axes[1]
x = np.arange(len(order))
width = 0.4
qwen_means = [NB11_QWEN[c] for c in order]
ax.bar(x - width/2, qwen_means, width, label='Qwen3-0.6B (NB11)', color='#cc3333', alpha=0.8, edgecolor='black')
ax.bar(x + width/2, means,      width, label='GPT-2-XL (NB14)',  color='#3366cc', alpha=0.8, edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(['skip','dry_run','random','real'])
ax.set_ylabel('A_displaced (mean)')
ax.set_title('Cross-scale comparison: does NB11 replicate?')
ax.legend(); ax.grid(alpha=0.3, axis='y'); ax.set_ylim(-0.05, 1.05)

fig.tight_layout()
fig.savefig(FIG_DIR / 'fig1_cross_scale.png', dpi=140)
plt.show()

### 14.9 Save and bundle

In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
summary = {
    'notebook': 'Notebook 14 — Cross-Scale Replication of NB11 on GPT-2-XL',
    'timestamp': ts,
    'model': MODEL_NAME,
    'n_pairs': len(pairs),
    'conditions': CONDITIONS,
    'data_source': 'pair indices same as NB11 (sourced from v3)',
    'per_condition_means': {c: float(df[df['condition']==c]['A_displaced'].mean()) for c in CONDITIONS},
    'per_condition_n': {c: int((df['condition']==c).sum()) for c in CONDITIONS},
    'pairwise_tests': {f'{a}_vs_{b}': v for (a,b), v in results.items()},
    'nb11_qwen_baseline': NB11_QWEN,
    'replication_check': {
        'content_independence_real_vs_random_p': float(real_v_random_p) if real_v_random_p is not None else None,
        'dryrun_safe_dry_vs_skip_p': float(dry_v_skip_p) if dry_v_skip_p is not None else None,
    },
}
with open(RESULTS_DIR / f'summary_nb14_{ts}.json', 'w') as f: json.dump(summary, f, indent=2)
df.to_csv(RESULTS_DIR / f'df_nb14_{ts}.csv', index=False)
print(json.dumps(summary, indent=2))

import shutil
bundle = f'nb14_results_{ts}'
shutil.make_archive(bundle, 'zip', RESULTS_DIR)
from google.colab import files
files.download(f'{bundle}.zip')

### What this tells us

**If the result replicates on GPT-2-XL** (real ≈ random_target, dry_run ≈ skip):

You can claim in the report:
> *"The optimizer-step mechanism for sequential edit damage is scale-invariant. Across both Qwen3-0.6B (n=10, NB11) and GPT-2-XL (n=10, NB14, 2.5x larger), random-target edits destroyed the prior edit indistinguishably from real edits, and dry-run gradients caused no damage. This indicates the finding is a general property of KL-stabilized gradient editing on transformer LMs, not an artifact of small-model dynamics."*

That's a *much* stronger claim than the single-scale version.

**If it doesn't replicate**, you have a different but still novel finding: the mechanism is scale-dependent in interesting ways, which connects to your existing Tables 2-4 cross-scale findings.

Either outcome strengthens the report.